# RAG Evaluation · 01 — 手动计算四大指标（纯 Python）

**本 Notebook 目标：** 不依赖任何评估框架，用 ~100 行纯 Python 实现 RAG 四大评估指标，建立计算直觉。

## 学习目标

- ✅ 理解四大指标的含义与公式：**Context Recall、Context Precision、Answer Relevancy、Faithfulness**
- ✅ 手动实现并运行每个指标的计算函数
- ✅ 用 DeepSeek LLM 作为 Faithfulness 裁判
- ✅ 读懂输出结果，诊断检索与生成的瓶颈
- ✅ 知道每个指标低分时该怎么优化

---
## 为什么要评估 RAG？

| 靠感觉的后果 | 数据驱动的能力 |
|---|---|
| "感觉这次答得不错" — 这不是工程 | **向老板汇报**：Faithfulness 从 0.61 → 0.84 |
| 换了 chunk size，效果变好了？**不确定** | **迭代方向明确**：Recall 低，说明 top_k 不够 |
| 上了 reranker，效果提升了？**说不清楚** | **版本对比量化**：v2 比 v1 Precision 高 12% |
| A/B 测试两个版本，选哪个？**无法决策** | **CI 中自动回归**：防止新版本悄悄变差 |

> **没有指标，优化 RAG 全是感觉游戏。**

---
## RAG 评估全景图

| | **找到了吗？（检索质量）** | **答对了吗？（生成质量）** |
|---|---|---|
| **完整性** | Context Recall | — |
| **精准性** | Context Precision | Answer Relevancy |
| **可信度** | — | Faithfulness |

**两个维度，四个象限：**

- **检索端**：找的内容够不够（Recall）、干不干净（Precision）
- **生成端**：答案切不切题（Relevancy）、有没有瞎编（Faithfulness）
- 四个指标缺一不可，只看一个会掩盖真正的问题

---
## 四大指标概览

| 指标 | 评估什么 | 公式简述 | 参考阈值 |
|------|---------|---------|---------|
| **Context Recall** | 该找的知识块都找到了吗 | \|GT ∩ Retrieved\| / \|GT\| | ≥ 0.80 |
| **Context Precision** | 找到的知识块都有用吗 | \|GT ∩ Retrieved\| / \|Retrieved\| | ≥ 0.75 |
| **Answer Relevancy** | 答案是否回答了问题 | 反向生成问题相似度均值 | ≥ 0.85 |
| **Faithfulness** | 答案有没有编造内容 | 被支持的 claim 比例 | ≥ 0.80 |

> GT = Ground Truth（标注的相关文档集合）；所有指标范围 [0, 1]，越高越好

---
# Part 1 · 手动计算四大指标（纯 Python，无框架）


### 环境准备

> **提示：** 请从上到下顺序运行 Part 1 的代码单元格，或使用 **Run All Above**。

In [1]:
import os, json, re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

DEEPSEEK_API_KEY = "sk-your-api-key-here"
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)
print(f"DeepSeek client initialized (key ...{DEEPSEEK_API_KEY[-4:]})")

DeepSeek client initialized (key ...9334)


### 示例知识库 & 查询

In [2]:
# 6 篇关于 AI/ML 主题的简短文档
KNOWLEDGE_BASE = [
    {"id": "doc_001", "content": (
        "Transformer 架构由 Vaswani 等人在 2017 年的论文《Attention Is All You Need》中提出。"
        "它完全基于注意力机制，摒弃了循环和卷积结构，在机器翻译任务上取得了突破性成果。"
    )},
    {"id": "doc_002", "content": (
        "检索增强生成（RAG）是一种将信息检索与大语言模型生成能力结合的技术。"
        "RAG 系统先从知识库中检索相关文档，再将其作为上下文提供给 LLM 生成答案，"
        "从而减少幻觉并提升答案准确性。"
    )},
    {"id": "doc_003", "content": (
        "向量数据库（Vector Database）用于存储和检索高维向量嵌入。"
        "常见的向量数据库包括 Pinecone、Weaviate、Milvus 和 Chroma。"
        "它们支持近似最近邻（ANN）搜索，适合语义相似度检索场景。"
    )},
    {"id": "doc_004", "content": (
        "大语言模型（LLM）的幻觉问题是指模型生成听起来合理但实际上不准确或虚构的内容。"
        "幻觉是 LLM 部署中的主要挑战之一，RAG 技术是缓解幻觉的有效手段之一。"
    )},
    {"id": "doc_005", "content": (
        "嵌入模型（Embedding Model）将文本映射到稠密向量空间，使语义相近的文本在向量空间中距离较近。"
        "常用的嵌入模型包括 OpenAI text-embedding-ada-002、BGE 系列和 E5 系列模型。"
    )},
    {"id": "doc_006", "content": (
        "提示工程（Prompt Engineering）是设计和优化输入提示以引导 LLM 产生期望输出的技术。"
        "常见技巧包括：少样本提示（Few-shot）、思维链提示（Chain-of-Thought）和角色扮演提示。"
    )},
]

QUERY = "什么是 RAG？它如何减少大语言模型的幻觉？"
GROUND_TRUTH_DOC_IDS = {"doc_002", "doc_004"}   # 应该找到的文档

# doc_002 正确检索到，doc_006 是噪声，doc_004 未检索到
RETRIEVED_DOCS = [KNOWLEDGE_BASE[1], KNOWLEDGE_BASE[5]]

GENERATED_ANSWER = (
    "RAG（检索增强生成）是将信息检索与大语言模型结合的技术。"
    "它先从知识库检索相关文档，再将文档作为上下文让 LLM 生成答案。"
    "通过提供真实文档作为依据，RAG 能有效减少 LLM 生成虚假信息的幻觉问题。"
)

print(f"查询: {QUERY}")
print(f"真实相关文档: {GROUND_TRUTH_DOC_IDS}")
print(f"检索到的文档: {[d['id'] for d in RETRIEVED_DOCS]}")
print(f"生成答案: {GENERATED_ANSWER}")

查询: 什么是 RAG？它如何减少大语言模型的幻觉？
真实相关文档: {'doc_004', 'doc_002'}
检索到的文档: ['doc_002', 'doc_006']
生成答案: RAG（检索增强生成）是将信息检索与大语言模型结合的技术。它先从知识库检索相关文档，再将文档作为上下文让 LLM 生成答案。通过提供真实文档作为依据，RAG 能有效减少 LLM 生成虚假信息的幻觉问题。


---
## 指标 1 · Context Recall（上下文召回率）

### 「该找的都找到了吗？」

**类比：考试复习**

> 如果期末考有 10 个知识点，你复习了 7 个，Recall = 0.7  
> 另外 3 个没复习到，考试时就会答不上来

**在 RAG 中：**
- Ground Truth = 回答这个问题**必须用到**的文档片段集合
- Retrieved = 检索器实际返回的文档片段集合
- **Recall 低 = 检索器「漏」了关键信息** → LLM 没有原材料，答不出来

**影响因素：** chunk size、top_k、embedding 模型、文档分块策略

$$\text{Context Recall} = \frac{|GT \cap Retrieved|}{|GT|}$$

| 问题 | 需要的文档片段（GT） | 实际检索到的片段 | Recall |
|------|---------------------|-----------------|--------|
| "公司退款政策？" | 片段 A、B、C | 片段 A、B、D | 2/3 ≈ **0.67** |
| "产品安装步骤？" | 片段 X、Y | 片段 X、Y、Z | 2/2 = **1.00** |

In [3]:
def compute_context_recall(retrieved_doc_ids: list, ground_truth_ids: set) -> float:
    """上下文召回率 = 检索到的相关文档数 / 所有真实相关文档数"""
    retrieved_set = set(retrieved_doc_ids)
    true_positives = len(retrieved_set & ground_truth_ids)
    return true_positives / len(ground_truth_ids) if ground_truth_ids else 0.0

retrieved_ids = [d['id'] for d in RETRIEVED_DOCS]
recall = compute_context_recall(retrieved_ids, GROUND_TRUTH_DOC_IDS)

print(f'Context Recall: {recall:.2f}')
print(f'  ✓ 检索到真实相关: {set(retrieved_ids) & GROUND_TRUTH_DOC_IDS}')
print(f'  ✗ 未检索到: {GROUND_TRUTH_DOC_IDS - set(retrieved_ids)}')

Context Recall: 0.50
  ✓ 检索到真实相关: {'doc_002'}
  ✗ 未检索到: {'doc_004'}


---
## 指标 2 · Context Precision（上下文精确率）

### 「找的都相关吗？」

**类比：搜索引擎质量**

> 搜索「北京天气」，结果第一条是广告，第二条是上海天气，第三条才是北京天气  
> 虽然最终找到了，但返回了很多噪声——Precision 低

**在 RAG 中：**
- **Precision 低 = 检索结果里混入太多噪声** → LLM 被干扰，容易答跑偏
- **影响因素：** Reranker、向量相似度阈值、关键词过滤、文档质量

$$\text{Context Precision} = \frac{|GT \cap Retrieved|}{|Retrieved|}$$

| 检索结果 | GT 中是否有 | 计入分子？ |
|---------|-----------|-----------|
| 片段 A：退款 30 天内 | ✅ 有 | 是 |
| 片段 B：到账 5-7 天 | ✅ 有 | 是 |
| 片段 C：支付方式介绍 | ❌ 无 | 否 |
| 片段 D：物流说明 | ❌ 无 | 否 |

> **Precision = 2 / 4 = 0.50** — 一半是噪声！

---
### Recall vs Precision 权衡

| 场景 | 优先保证 | 可以牺牲 | 原因 |
|------|---------|---------|------|
| 医疗/法律问答 | **Recall** | Precision | 漏掉关键信息代价太高 |
| 客服 FAQ | **Precision** | Recall | 答案要简洁、不混淆用户 |
| 学术研究助手 | 两者都要 | — | 全面且准确 |
| 实时搜索增强 | **Precision** | Recall | 速度优先，宁缺毋滥 |

**调参方向：**
- 提高 Recall → 增大 `top_k`、换更强的 embedding、加混合检索
- 提高 Precision → 加 Reranker、提高相似度阈值、改善 chunk 质量
- **两者同时低** = embedding 模型根本没学好，考虑换模型

In [4]:
def compute_context_precision(retrieved_doc_ids: list, ground_truth_ids: set) -> float:
    """上下文精确率 = 检索到的相关文档数 / 检索到的所有文档数"""
    if not retrieved_doc_ids:
        return 0.0
    retrieved_set = set(retrieved_doc_ids)
    true_positives = len(retrieved_set & ground_truth_ids)
    return true_positives / len(retrieved_doc_ids)

precision = compute_context_precision(retrieved_ids, GROUND_TRUTH_DOC_IDS)

print(f'Context Precision: {precision:.2f}')
print(f'  共检索 {len(RETRIEVED_DOCS)} 个文档，其中 {int(precision * len(RETRIEVED_DOCS))} 个真正相关')

Context Precision: 0.50
  共检索 2 个文档，其中 1 个真正相关


---
## 指标 3 · Answer Relevancy（答案相关性）

### 「答案是否切题？」

> **重要区分：Relevancy ≠ Correctness**

| | Relevancy（相关性） | Correctness（正确性） |
|---|---|---|
| 定义 | 答案有没有回答问题 | 答案内容是否事实正确 |
| 低分原因 | 答非所问、跑题 | 内容错误、数据有误 |
| 谁来判断 | 问题与答案的相似度 | 需要对比 ground truth |

**例子：**
- 问："退款要多久？"
- 差答案："我们提供优质的售后服务，请放心购物。" → **Relevancy 低**（没回答）
- 好答案："退款需要 5-7 个工作日。" → **Relevancy 高**

---

In [5]:
import re

def compute_answer_relevancy_proxy(query: str, answer: str) -> float:
    """
    答案相关性的简化代理版本：使用问题与答案之间的关键词 Jaccard 相似度。
    注意：真实的 Answer Relevancy 需要嵌入模型计算语义相似度。
    """
    def tokenize(text: str) -> set:
        tokens = re.findall(r"[一-鿿]+|[a-zA-Z0-9]+", text.lower())
        return set(tokens)

    query_tokens = tokenize(query)
    answer_tokens = tokenize(answer)
    intersection = len(query_tokens & answer_tokens)
    union = len(query_tokens | answer_tokens)
    return intersection / union if union > 0 else 0.0

relevancy = compute_answer_relevancy_proxy(QUERY, GENERATED_ANSWER)
print(f'Answer Relevancy (Jaccard proxy): {relevancy:.2f}')
print('  注意：真实评估需要嵌入模型，此处使用 Jaccard 关键词重叠作为近似')

Answer Relevancy (Jaccard proxy): 0.08
  注意：真实评估需要嵌入模型，此处使用 Jaccard 关键词重叠作为近似


---
## 指标 4 · Faithfulness（忠实度）

### 「有没有瞎编？」

**幻觉（Hallucination）是 RAG 最大的生产风险**

**什么是 Faithfulness 低？**  
LLM 生成了**上下文里没有**的内容：
- 捏造数字、日期、政策条款
- 混淆相似但不同的信息
- 用训练知识「补全」而非用检索内容回答

**为什么 RAG 也会幻觉？**
- 检索到的上下文不够充分
- LLM 置信度高时倾向于「补充」
- System prompt 约束不够强
- 多跳推理时中间步骤出错

> **Faithfulness = 答案中有多少比例的声明，可以在检索到的上下文中找到支撑**

---
### Claim 提取 → 逐条验证流程（LLM-as-Judge）

```
RAG 答案
  │
  ▼
Step 1: LLM 提取原子级 Claim（每个独立事实一条）
  ├── Claim 1: "退款申请期限为 30 天"
  ├── Claim 2: "退款 5-7 个工作日到账"
  └── Claim 3: "退款不支持部分退款"   ← 上下文里没有！
  │
  ▼
Step 2: 每个 Claim 与 Retrieved Context 逐条比对
  ├── Claim 1: ✅ 支持（上下文第 2 段有明确表述）
  ├── Claim 2: ✅ 支持（上下文第 3 段）
  └── Claim 3: ❌ 无支撑（上下文未提及）
  │
  ▼
Faithfulness = 支持的 Claim 数 / 总 Claim 数 = 2/3 ≈ 0.67
```

In [6]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI



_api_key = "sk-your-api-key-here"
client = OpenAI(api_key=_api_key, base_url="https://api.deepseek.com")
print(f"DeepSeek client initialized (key ...{_api_key[-4:]})")

def compute_faithfulness_llm_judge(answer: str, context_docs: list) -> tuple:
    """
    忠实度：验证答案中的每个声明是否都能在检索上下文中找到支撑。
    使用 DeepSeek LLM 作为裁判，输出 JSON 格式的逐条验证结果。
    """
    context_text = "\n\n".join(
        [f"[文档 {i+1}] {doc['content']}" for i, doc in enumerate(context_docs)]
    )

    prompt = f"""你是一个严格的 RAG 评估裁判。

请判断以下【答案】中的每个声明（claim）是否都有【上下文】支持。

【上下文】:
{context_text}

【答案】:
{answer}

请按以下 JSON 格式返回结果（只返回 JSON，不要有其他文字）：
{{
  \"claims\": [
    {{\"claim\": \"声明内容\", \"supported\": true}},
    {{\"claim\": \"声明内容\", \"supported\": false}}
  ],
  \"faithfulness_score\": 0.0
}}

faithfulness_score = 有上下文支持的声明数 / 总声明数，范围 0.0 到 1.0。"""

   

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    raw = response.choices[0].message.content.strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    result = json.loads(raw.strip())
    return float(result.get('faithfulness_score', 0.0)), result.get('claims', [])

print('正在调用 DeepSeek LLM 评估 Faithfulness...')
faithfulness, claims = compute_faithfulness_llm_judge(GENERATED_ANSWER, RETRIEVED_DOCS)
print(f'Faithfulness Score: {faithfulness:.2f}')
print('\n逐条验证结果:')
for item in claims:
    status = '✓ 有支撑' if item.get('supported') else '✗ 无支撑'
    print(f'  {status} - {item.get("claim", "")}')

DeepSeek client initialized (key ...4fb1)
正在调用 DeepSeek LLM 评估 Faithfulness...
Faithfulness Score: 1.00

逐条验证结果:
  ✓ 有支撑 - RAG（检索增强生成）是将信息检索与大语言模型结合的技术。
  ✓ 有支撑 - 它先从知识库检索相关文档，再将文档作为上下文让 LLM 生成答案。
  ✓ 有支撑 - 通过提供真实文档作为依据，RAG 能有效减少 LLM 生成虚假信息的幻觉问题。


---
### 汇总：四大指标计算结果

In [27]:
print('=' * 60)
print('  汇总 / Summary')
print('=' * 60)
print(f'  Context Recall:    {recall:.2f}  (目标 > 0.80)')
print(f'  Context Precision: {precision:.2f}  (目标 > 0.75)')
print(f'  Answer Relevancy:  {relevancy:.2f}  (Jaccard 代理，目标 > 0.85)')
print(f'  Faithfulness:      {faithfulness:.2f}  (目标 > 0.80)')
print('=' * 60)
print()
print('诊断：')
if recall < 0.8:  print(f'  ⚠️  Recall {recall:.2f} < 0.80 → 检索漏了关键文档（doc_004 未找到）')
if precision < 0.75: print(f'  ⚠️  Precision {precision:.2f} < 0.75 → 检索结果含噪声（doc_006 无关）')

  汇总 / Summary
  Context Recall:    0.50  (目标 > 0.80)
  Context Precision: 0.50  (目标 > 0.75)
  Answer Relevancy:  0.08  (Jaccard 代理，目标 > 0.85)
  Faithfulness:      1.00  (目标 > 0.80)

诊断：
  ⚠️  Recall 0.50 < 0.80 → 检索漏了关键文档（doc_004 未找到）
  ⚠️  Precision 0.50 < 0.75 → 检索结果含噪声（doc_006 无关）


---
# Part 2 · 指标诊断与优化

---
# Part 3 · 指标诊断与优化

---
## 指标低了怎么办 — 诊断表

| 指标低 | 阈值参考 | 可能原因 | 优化方向 |
|--------|---------|---------|---------|
| **Context Recall** | < 0.75 | top_k 太小；chunk 太大；embedding 召回差 | 加大 top_k；缩小 chunk_size；换更强 embedding |
| **Context Precision** | < 0.65 | 检索结果含大量无关片段 | 加 Reranker；提高相似度阈值；改善文档质量 |
| **Answer Relevancy** | < 0.80 | Prompt 没有引导模型直接回答；答案太长绕弯 | Query 重写；优化 response prompt |
| **Faithfulness** | < 0.75 | System prompt 约束弱；检索内容不足 | 强化 system prompt；先提高 Recall |

> **优先级建议：先修 Recall → 再修 Faithfulness → 再修 Precision → 最后 Relevancy**

---
## Context Recall 低 — 怎么修

**根本原因：检索器没找到回答问题所需的关键信息**

```python
# 1. 增大 top_k
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 从 3 改到 6

# 2. 缩小 chunk_size，增加 overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,      # 从 1000 缩小
    chunk_overlap=128,   # 从 0 增加
)

# 3. 混合检索：向量 + BM25 关键词
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

# 4. 换更强的 Embedding 模型
# text-embedding-ada-002 → text-embedding-3-large
# 或中文专用：BAAI/bge-large-zh-v1.5
```

---
## Context Precision 低 — 怎么修

**根本原因：检索结果混入了太多与问题无关的噪声片段**

```python
# 方案 1：加 Reranker 二次排序
from langchain.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-multilingual-v3.0", top_n=3)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# 方案 2：提高相似度阈值，过滤低质量结果
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.75, "k": 6}
)
```

---
## Faithfulness 低 — 怎么修

**根本原因：LLM 在生成答案时脱离了检索内容，加入了自己的「发挥」**

```python
# 方案 1：强化 System Prompt 约束
"""你是一个严格基于文档的问答助手。

规则：
1. 只使用【参考文档】中的信息来回答问题
2. 如果文档中没有相关信息，回答"根据现有文档，无法回答此问题"
3. 不要添加文档中没有的任何信息
4. 引用原文时保持准确，不要改写核心事实

【参考文档】
{context}

用户问题：{question}"""

# 方案 2：先提高 Recall（LLM 有足够原材料就不需要「补充」）
# 方案 3：降低 LLM temperature
llm = ChatOpenAI(model="gpt-4o", temperature=0)
```

---
## Answer Relevancy 低 — 怎么修

**根本原因：答案没有直接回答用户的问题（答非所问或绕弯太多）**

```python
# 方案 1：Query 重写 — 让问题更精确
"""将以下用户问题改写为更清晰、更具体的检索查询：
原始问题：{question}
改写后的查询（直接给出，不要解释）："""

# 方案 2：HyDE（假设文档嵌入）— 先生成假设答案再检索
"""请为以下问题写一个假设性的简短答案（50字以内）：
问题：{question}"""
# 用假设答案的 embedding 去检索，比问题 embedding 更接近答案空间

# 方案 3：优化 Response Prompt，要求直接回答
"""请直接回答用户的问题。
第一句话必须直接给出答案，不要铺垫。
问题：{question}
上下文：{context}"""
```

---
# Part 4 · 进阶：生产环境评估

---
## 常见错误 Top 5

**No.1 没有 Ground Truth 就跑评估**
> Recall 和 Precision 必须有标注的参考上下文，否则结果没有意义。至少准备 50 条标注数据。

**No.2 用同一个 LLM 既生成答案又评估**
> 自己评自己的分必然虚高。评估 LLM 应比生成 LLM 更强，或使用独立的评估模型。

**No.3 只看 Faithfulness，忽略 Recall**
> Faithfulness 高但 Recall 低 = 答案很诚实但信息不全。用户会觉得「答得不够」。

**No.4 只看平均分，不看样本分布**
> 平均 0.80 可能藏着 30% 的样本 Faithfulness = 0.0（完全幻觉）。务必看 min、p10 分位数。

**No.5 评估集太小（< 20 条）**
> 样本不足时指标波动极大，换几条样本分数可能差 0.2+。生产系统最少 100 条。

---
# 关键 Takeaways

1. **量化是工程的基础** — 没有指标，优化 RAG 全是感觉游戏
2. **四个指标覆盖两个维度** — 检索端（Recall + Precision）和生成端（Relevancy + Faithfulness）
3. **优化顺序**：先 Recall → 再 Faithfulness → 再 Precision → 最后 Relevancy
4. **手动实现能建立直觉** — 理解公式后再用框架，不会被黑盒误导

## 课程小结

| 指标 | 核心问题 | 公式 | 低分怎么修 |
|------|---------|------|----------|
| **Context Recall** | 该找的都找到了吗 | \|GT∩Ret\|/\|GT\| | 加 top_k、缩 chunk、换 embedding |
| **Context Precision** | 找的都相关吗 | \|GT∩Ret\|/\|Ret\| | 加 Reranker、提相似度阈值 |
| **Answer Relevancy** | 答的切题吗 | 反向问题相似度均值 | Query 重写、优化 prompt |
| **Faithfulness** | 有没有瞎编 | 被支持的 claim 比例 | 强化 system prompt、先提 Recall |

> **下一步：** 用 `02_ragas_quickstart.ipynb` 学习如何用 RAGAS 框架自动化这些评估